In [1]:
#!pip install dtreeviz graphviz ipykernel cairosvg

In [2]:
import dtreeviz
import graphviz

In [3]:
import os

graphviz_bin = r'C:/Program Files/Graphviz/bin'

os.environ['PATH'] = graphviz_bin + os.pathsep + os.environ['PATH']

In [4]:
import matplotlib as mpl
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

mpl.rcParams['svg.fonttype'] = 'none'

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

plt.style.use("ggplot")

In [6]:
df = pd.read_csv('jeju_bus.csv')

display(df.head())

print('shape:', df.shape)

print('전체 결측치 개수:', df.isnull().sum().sum())

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


shape: (210457, 14)
전체 결측치 개수: 0


In [7]:
df_model = df.copy()

df_model['original_index'] = df_model.index
target_col = 'next_arrive_time'

In [8]:
df_model['date'] = pd.to_datetime(df_model['date'])

df_model['day'] = df_model['date'].dt.day
df_model['dayofweek'] = df_model['date'].dt.dayofweek

df_model['now_hour'] = df_model['now_arrive_time'].astype(str).str.extract(r'(\d+)').astype(float)

df_model[['date', 'day', 'dayofweek', 'now_arrive_time', 'now_hour']].head()

,date,day,dayofweek,now_arrive_time,now_hour
0,2019-10-15,15,1,06시,6.0
1,2019-10-15,15,1,06시,6.0
2,2019-10-15,15,1,06시,6.0
3,2019-10-15,15,1,06시,6.0
4,2019-10-15,15,1,07시,7.0


df_model['station_segment'] = (
    df_model['now_station'].astype(str) + ' → ' + df_model['next_station'].astype(str)
)
df_model.head()

print('station_segment 고유값 개수:', df_model['station_segment'].nunique())

In [9]:
def calculate_distance_km(lat1, lon1, lat2, lon2):
    # 위도/경도 사이의 거리를 km 단위로 계산!
    # 1: 현재 내 정류징
    # 2: 각 주요 지점
    earth_radius_km = 6371

    # radius: 반지름, 반
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # 거리 계산 공식
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2        
    )
    c = 2 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c

In [10]:
reference_points = {
    'up'    : (33.506286, 126.490312),
    'down'  : (33.246742, 126.562387),
    'right' : (33.493521, 126.895326),
    'center': (33.379724, 126.545315)
}

In [11]:
def assign_region_info(data, lat_col, lon_col):
    up_lat, up_lon = reference_points['up']
    down_lat, down_lon = reference_points['down']
    right_lat, right_lon = reference_points['right']
    center_lat, center_lon = reference_points['center']

    distance_to_up = calculate_distance_km(data[lat_col], data[lon_col], up_lat, up_lon)
    distance_to_down = calculate_distance_km(data[lat_col], data[lon_col], down_lat, down_lon)
    distance_to_right = calculate_distance_km(data[lat_col], data[lon_col], right_lat, right_lon)
    distance_to_center = calculate_distance_km(data[lat_col], data[lon_col], center_lat, center_lon)

    distance_table = pd.DataFrame({
        'up': distance_to_up,
        'down': distance_to_down,
        'right': distance_to_right,
        'center': distance_to_center        
    }, index = data.index)

    nearest_region = distance_table.idxmin(axis = 1)

    result = pd.DataFrame({
        'dist_name': nearest_region,
        'dist_to_up': distance_to_up,
        'dist_to_down': distance_to_down,
        'dist_to_right': distance_to_right,
        'dist_to_center': distance_to_center
    }, index = data.index)

    return result

In [12]:
now_region_info = assign_region_info(
    df_model,
    'now_latitude',
    'now_longitude'    
)

now_region_info.head()

,dist_name,dist_to_up,dist_to_down,dist_to_right,dist_to_center
0,up,7.962505,23.319056,32.135034,8.532122
1,up,8.003869,23.473015,31.905930,8.710701
2,up,8.158338,23.582519,31.583874,8.861671
3,up,5.774762,25.961685,32.635035,11.118256
4,up,2.332803,27.295447,37.141639,12.673969


In [13]:
next_region_info = assign_region_info(
    df_model,
    'next_latitude',
    'next_longitude'    
)

next_region_info.head()

,dist_name,dist_to_up,dist_to_down,dist_to_right,dist_to_center
0,up,8.003869,23.473015,31.905930,8.710701
1,up,8.158338,23.582519,31.583874,8.861671
2,up,8.387595,23.701416,31.175511,9.041978
3,up,5.429627,26.539110,32.693959,11.692688
4,up,2.276139,27.400941,37.514443,12.832868


In [14]:
df_model['now_dist_name'] = now_region_info['dist_name']
df_model['now_dist_to_up'] = now_region_info['dist_to_up']
df_model['now_dist_to_down'] = now_region_info['dist_to_down']
df_model['now_dist_to_right'] = now_region_info['dist_to_right']
df_model['now_dist_to_center'] = now_region_info['dist_to_center']

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,...,next_arrive_time,original_index,day,dayofweek,now_hour,now_dist_name,now_dist_to_up,now_dist_to_down,now_dist_to_right,now_dist_to_center
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,...,24,0,15,1,6.0,up,7.962505,23.319056,32.135034,8.532122
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,...,36,1,15,1,6.0,up,8.003869,23.473015,31.905930,8.710701
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,...,40,2,15,1,6.0,up,8.158338,23.582519,31.583874,8.861671
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,...,42,3,15,1,6.0,up,5.774762,25.961685,32.635035,11.118256
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,...,64,4,15,1,7.0,up,2.332803,27.295447,37.141639,12.673969


In [15]:
df_model['next_dist_name'] = next_region_info['dist_name']
df_model['next_dist_to_up'] = next_region_info['dist_to_up']
df_model['next_dist_to_down'] = next_region_info['dist_to_down']
df_model['next_dist_to_right'] = next_region_info['dist_to_right']
df_model['next_dist_to_center'] = next_region_info['dist_to_center']

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,...,now_dist_name,now_dist_to_up,now_dist_to_down,now_dist_to_right,now_dist_to_center,next_dist_name,next_dist_to_up,next_dist_to_down,next_dist_to_right,next_dist_to_center
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,...,up,7.962505,23.319056,32.135034,8.532122,up,8.003869,23.473015,31.905930,8.710701
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,...,up,8.003869,23.473015,31.905930,8.710701,up,8.158338,23.582519,31.583874,8.861671
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,...,up,8.158338,23.582519,31.583874,8.861671,up,8.387595,23.701416,31.175511,9.041978
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,...,up,5.774762,25.961685,32.635035,11.118256,up,5.429627,26.539110,32.693959,11.692688
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,...,up,2.332803,27.295447,37.141639,12.673969,up,2.276139,27.400941,37.514443,12.832868


In [16]:
df_model['dist_segment_name'] = (
    df_model['now_dist_name'].astype(str)
    + ' → '
    + df_model['next_dist_name'].astype(str)    
)

print('dist_segment_name 고유값 개수: ', df_model['dist_segment_name'].nunique())
df_model['dist_segment_name'].value_counts()

dist_segment_name 고유값 개수:  12


dist_segment_name
up → up            121131
down → down         48377
right → right       34032
center → center      5512
center → up           321
right → up            276
up → center           207
down → right          144
up → right            137
right → down          134
down → center          95
center → down          91
Name: count, dtype: int64

In [17]:
df_model.columns.tolist()

['id',
 'date',
 'route_id',
 'vh_id',
 'route_nm',
 'now_latitude',
 'now_longitude',
 'now_station',
 'now_arrive_time',
 'distance',
 'next_station',
 'next_latitude',
 'next_longitude',
 'next_arrive_time',
 'original_index',
 'day',
 'dayofweek',
 'now_hour',
 'now_dist_name',
 'now_dist_to_up',
 'now_dist_to_down',
 'now_dist_to_right',
 'now_dist_to_center',
 'next_dist_name',
 'next_dist_to_up',
 'next_dist_to_down',
 'next_dist_to_right',
 'next_dist_to_center',
 'dist_segment_name']

In [18]:
selected_numeric_features = [
     'distance', 
     'day',  
     'dayofweek',
     'now_hour',
     'now_dist_to_up',
     'now_dist_to_down',
     'now_dist_to_right',
     'now_dist_to_center',
     'next_dist_to_up',
     'next_dist_to_down',
     'next_dist_to_right',
     'next_dist_to_center'
]

In [19]:
selected_categorical_features = [
     'route_nm',
     'now_station',
     'next_station',
     'dist_segment_name'
]

In [20]:
selected_features = selected_numeric_features + selected_categorical_features
selected_features

['distance',
 'day',
 'dayofweek',
 'now_hour',
 'now_dist_to_up',
 'now_dist_to_down',
 'now_dist_to_right',
 'now_dist_to_center',
 'next_dist_to_up',
 'next_dist_to_down',
 'next_dist_to_right',
 'next_dist_to_center',
 'route_nm',
 'now_station',
 'next_station',
 'dist_segment_name']

In [21]:
upper_1pct = df_model[target_col].quantile(0.99)

upper_1pct

np.float64(340.0)

In [22]:
df_model_selected = df_model [ df_model[target_col] <= upper_1pct ].copy()

In [23]:
X = df_model_selected[selected_features]

In [24]:
y = df_model_selected[target_col]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
                                         X, y, 
                                         test_size = 0.2,
                                         random_state = 42
                                    )

In [26]:
preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown = 'ignore'), selected_categorical_features)
        ], remainder = 'passthrough'
    )

In [27]:
def evaluate_regression_model(model_name, experiment, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_ture, y_pred))

    print(f'[{model_name}] MAE : {mae:.4f} | RMSE : {rmse:.4f}')

    return {'model_name': model_name, 'description': description,
            'MAE': mae, 'RMSE' : rmse
    }

In [31]:
def train_xgb_pipeline(model_name1, description1, xgb_params):
    xgb_model = XGBRegressor(
        objective = 'reg:squarederror',
        random_state = 42,
        n_jobs = -1,# 가용 CPU 최대 사용
        **xgb_params
    )

    model_pipeline = Pipeline(steps=[
                             ('preprocessor', preprocessor),
                             ('model', xgb_model)
                     ])

    model_pipeline.fit(X_train, y_train)
    y_pred1 = model_pipeline.predict(X_test)

    evaluation = evaluation_regression_model(
                                model_name=model_name1,
                                description = description1,
                                y_true = y_test,
                                y_pred = y_pred1
                                )
    return{"model_name": model_name1, 'pipeline': model_pipeline,
          'ypred':y_pred1, 'evaluation': evaluation}                             
    

In [44]:
param_di = {
    'model__max_depth':[3, 5, 7, 9],# 트리 깊이
    'model__learning_rate': [0.03, 0.05, 0.1],# 학습률(각 트리 보정 반영 비율)
    'model__n_estimators': [200, 300, 400]# 트리 갯수
}

scoring = 'neg_mean_abosolute_error'

{'model__max_depth': [3, 5, 7, 9],
 'model__learning_rate': [0.03, 0.05, 0.1],
 'model__n_estimators': [200, 300, 400]}

In [42]:
search_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=1        
    ))    
])

In [48]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator = search_model,
    param_distributions = param_di,
    n_iter = 25,# 시도할 조합 수(실행이 오래 걸리면 10으로 줄일 수 있음)
    scoring = 'neg_mean_absolute_error',# 음수 MAE(클수록 좋음)
    cv = 5,# train을 5개로 나눠 내부 교차검증
    random_state = 42,
    n_jobs = -1,
    verbose = 1# 진행 상황 출력
)    

In [49]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [3, 5, ...], 'model__n_estimators': [200, 300, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used he